# Tutorial 6 : Shape Characterization

In [ ]:
!pip install cellcharter
!pip install squidpy
!pip install scanpy

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of dask[array] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of ome-zarr to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
import squidpy as sq
import cellcharter as cc
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

flag = True
df_val = pd.DataFrame()

Mounting google drive to accessing input data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Shape Characterization

In [ ]:
def cellcharter_metrics(data_name, category, data_path, ctp_path):

    global flag, df_val

    adata = sc.read_h5ad(data_path)
    ctp = pd.read_csv(ctp_path)
    print ("ctp shape", ctp.shape, "and", ctp.columns)

    if (adata.shape[0] != ctp.shape[0]):
        adata = adata[adata.obs_names.isin(ctp[ctp.columns[0]])]

    print ("adata shape", adata.shape)

    for c in ctp.columns[1:]:
        threshold = np.median(ctp[c].values)
        adata.obs[c + "_spatial_cluster"] = (ctp[c] > threshold).astype(int).values

    sq.gr.spatial_neighbors(adata, coord_type = 'generic', delaunay = True) # library_key='sample',

    for c in ctp.columns[1:]:

        cc.gr.connected_components(adata, cluster_key = c + '_spatial_cluster')
        cc.tl.boundaries(adata, min_hole_area_ratio = 0.1)
        cc.tl.linearity(adata)
        cc.tl.curl(adata)
        cc.tl.elongation(adata)

        adata.obs.rename(columns = {'component': c +'_component'}, inplace = True)
        #print (adata)
        mapping = pd.DataFrame({"Act" : adata.obs[c + "_spatial_cluster"].values, "Pred" : adata.obs[c + "_component"].values})
        mapping = mapping.drop_duplicates()

        #adata.uns[c + '_shape_component'] = adata.uns.pop('shape_component')

        t = mapping.drop_duplicates()
        t = t[t["Act"] == 1]
        t.dropna(inplace = True)

        comp_itr = 1
        for comp in t["Pred"].values:

            linearity = adata.uns["shape_component"]["linearity"][comp]
            curl = adata.uns["shape_component"]["curl"][comp]
            elongation = adata.uns["shape_component"]["elongation"][comp]

            if (flag):
                df_val["Category"] = [category]
                df_val["Dataset"] = [data_name]
                df_val["cell_type"] = [c + "_" + str(comp_itr)]
                df_val["linearity"] = [linearity]
                df_val["curl"] = [curl]
                df_val["elongation"] = [elongation]

                #print (df_val)
                comp_itr += 1
                flag = False
            else:
                new_row = {"Category" : category, "Dataset" : data_name, "cell_type": c + "_" + str(comp_itr), "linearity" : linearity, "curl" : curl, "elongation" : elongation}
                comp_itr += 1
                #df_val = df_val.append(new_row, ignore_index = True)
                df_val = pd.concat([df_val, pd.DataFrame([new_row])], ignore_index=True)
                #print (df_val)


In [ ]:
lst = {
        "Brain" : ["151508"], # "151673", "151508", "151509", "151669", "151670"
      "Cancer": [],
      "Development" : []
        }

path = "/content/drive/MyDrive/Major_project/Benchmarking_Shared/spDDB_tutorials/"

for l in lst["Brain"]:

    data_path = path + "1_data/synthetic_spatial_gene_exp/simulated_spatial_gene_expression.h5ad"
    ctp_path = path + "4_data/Simulated_cell_type_proportion_DLPFC_151508.csv"

    cellcharter_metrics(l, "Brain", data_path, ctp_path)

ctp shape (4382, 18) and Index(['Unnamed: 0', 'AST_FB', 'AST_PP', 'Endothelial', 'IN_PV', 'IN_SST',
       'IN_SV2C', 'IN_VIP', 'L2_3', 'L4', 'L5_6', 'L5_6_CC', 'Microglia',
       'Neu_NRGN_I', 'Neu_NRGN_II', 'Neu_mat', 'OPC', 'Oligodendrocytes'],
      dtype='object')
adata shape (4382, 11403)
INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        


/tmp/ipykernel_3161/929752537.py:24: FutureWarning: linearity is deprecated and will be removed in the next release. Please use `linearity_metric` instead.
  cc.tl.linearity(adata)
/tmp/ipykernel_3161/929752537.py:25: FutureWarning: curl is deprecated and will be removed in the next release. Please use `curl_metric` instead.
  cc.tl.curl(adata)
/tmp/ipykernel_3161/929752537.py:26: FutureWarning: elongation is deprecated and will be removed in the next release. Please use `elongation_metric` instead.
  cc.tl.elongation(adata)
/tmp/ipykernel_3161/929752537.py:24: FutureWarning: linearity is deprecated and will be removed in the next release. Please use `linearity_metric` instead.
  cc.tl.linearity(adata)
/tmp/ipykernel_3161/929752537.py:25: FutureWarning: curl is deprecated and will be removed in the next release. Please use `curl_metric` instead.
  cc.tl.curl(adata)
/tmp/ipykernel_3161/929752537.py:26: FutureWarning: elongation is deprecated and will be removed in the next release. Plea

Specify metrics

In [ ]:
metric = "linearity" # "elongation" # "curl"

In [ ]:
def consider_all_comps(ct_lst):

    new_ct_lst = []

    for b in ct_lst:
        b_clip = b[:-1]
        new_ct_lst += [b]

        last_char = b[-1]

        new_ct_lst += [b_clip + str(d) for d in range(1, int(last_char))]

    return new_ct_lst

def get_top_bottom_25(df_subset, per, m):

    df_subset = df_subset[["cell_type", m]]
    df_subset = df_subset.groupby('cell_type')[m].median().reset_index()

    # Rename the columns for clarity (optional)
    df_subset.columns = ['cell_type', m]

    sorted_df = df_subset.sort_values(by = m).reset_index(drop = True)
    bottom_cts = sorted_df[: round(per*len(sorted_df))][["cell_type", m]]

    per_neg = 1 - per
    top_cts = sorted_df[round(per_neg*len(sorted_df)) : ][["cell_type", m]]
    #print (bottom_cts, top_cts)

    new_bottom_cts = consider_all_comps(bottom_cts["cell_type"].values)
    new_top_cts = consider_all_comps(top_cts["cell_type"].values)
    return new_bottom_cts, new_top_cts

Top and Bottom cell types with chosen shape metrics for Brain datasets

In [ ]:
print (df_val)

   Category Dataset           cell_type  linearity      curl  elongation
0     Brain  151508            AST_FB_1   0.763348  0.110810    0.110016
1     Brain  151508            AST_FB_1   0.763348  0.110810    0.110016
2     Brain  151508            AST_PP_1   1.000000  0.000000    0.345660
3     Brain  151508            AST_PP_2   1.000000  0.101009    0.653668
4     Brain  151508       Endothelial_1   0.941555  0.556208    0.052118
5     Brain  151508             IN_PV_1   0.576744  0.343922    0.000154
6     Brain  151508            IN_SST_1   0.726808  0.000000    0.052081
7     Brain  151508           IN_SV2C_1   0.913603  0.000000    0.345682
8     Brain  151508            IN_VIP_1   1.000000  0.000000    0.331758
9     Brain  151508              L2_3_1   0.798326  0.032302    0.150539
10    Brain  151508                L4_1   1.000000  0.000000    0.414979
11    Brain  151508              L5_6_1   1.000000  0.000000    0.471710
12    Brain  151508           L5_6_CC_1   1.000000 

In [ ]:
### Linearity
df_brain = df_val[df_val["Category"] == "Brain"]
dfs = []

Organs = {
    "DLPFC" : ['151508'] # '151674', '151508', '151509', '151669', '151670'
    #"Mouse_brain" : ['Mouse_brain_ST48', 'Mouse_brain_ST52']
}

for key in Organs.keys():

    data = Organs[key]
    #print (data)
    df_subset = df_brain[df_brain["Dataset"].isin(data)].reset_index(drop = True)

    #print (df_subset.shape)
    bottom_cts, top_cts = get_top_bottom_25(df_subset, 0.25, metric)
    print ("Bottom", bottom_cts, "\n Top", top_cts, "\n\n\n")

    df_bottom = df_subset[df_subset["cell_type"].isin(bottom_cts)].reset_index(drop = True)
    df_bottom["type"] = "bottom"
    dfs += [df_bottom]

    df_top = df_subset[df_subset["cell_type"].isin(top_cts)].reset_index(drop = True)
    df_top["type"] = "top"
    dfs += [df_top]

df_shape_brain = pd.concat(dfs, axis = 0).reset_index(drop = True)

Bottom ['IN_PV_1', 'IN_SST_1', 'Neu_mat_1', 'AST_FB_1', 'Microglia_1'] 
 Top ['AST_PP_2', 'AST_PP_1', 'Neu_NRGN_II_1', 'L4_1', 'OPC_1', 'Oligodendrocytes_1'] 





In [ ]:
print (df_shape_brain)

   Category Dataset           cell_type  linearity      curl  elongation  \
0     Brain  151508            AST_FB_1   0.763348  0.110810    0.110016   
1     Brain  151508            AST_FB_1   0.763348  0.110810    0.110016   
2     Brain  151508             IN_PV_1   0.576744  0.343922    0.000154   
3     Brain  151508            IN_SST_1   0.726808  0.000000    0.052081   
4     Brain  151508         Microglia_1   0.795943  0.000000    0.396660   
5     Brain  151508           Neu_mat_1   0.747737  0.000000    0.144467   
6     Brain  151508            AST_PP_1   1.000000  0.000000    0.345660   
7     Brain  151508            AST_PP_2   1.000000  0.101009    0.653668   
8     Brain  151508                L4_1   1.000000  0.000000    0.414979   
9     Brain  151508       Neu_NRGN_II_1   1.000000  0.000000    0.462463   
10    Brain  151508               OPC_1   1.000000  0.170473    0.365133   
11    Brain  151508  Oligodendrocytes_1   1.000000  0.006502    0.427947   

      type 